# Tarea 1 - Procesamiento de Lenguaje Natural\n## Pipeline básico: Limpieza, Tokenización, Clasificación de Comentarios\n\n**Objetivo:** Leer comentarios de estudiantes, limpiar el texto, tokenizar, clasificar por sentimiento y categoría, y guardar los resultados en un archivo CSV.

## 1. Importación de librerías

In [1]:
import sys
print(f"Python version: {sys.version}")

from pathlib import Path
import csv
import re
import unicodedata
from collections import Counter
import pandas as pd

Python version: 3.12.6 (main, Oct 11 2025, 10:55:08) [Clang 17.0.0 (clang-1700.3.19.1)]


## 2. Definición de diccionarios (stopwords, palabras positivas/negativas, categorías)

In [2]:
ARCHIVO_ENTRADA = Path("comentarios_clase1.txt")
ARCHIVO_SALIDA = Path("resultados_clase1.csv")

STOPWORDS = {
    "el", "la", "los", "las", "un", "una", "unos", "unas",
    "de", "del", "y", "o", "pero", "aunque", "a", "en", "con",
    "por", "para", "me", "mi", "mis", "es", "fue", "muy", "que",
    "se", "al", "lo", "no", "su", "sus", "tuvo", "durante"
}

PALABRAS_POSITIVAS = {
    "excelente", "bien", "buena", "bueno", "interesante", "util", "gusto",
    "claro", "claridad", "paciencia", "relevante", "ayudo", "facil", "organizado"
}

PALABRAS_NEGATIVAS = {
    "demasiadas", "rapido", "faltaron", "fallo", "confusos", "confuso",
    "poca", "lenta", "mal", "problema", "cayo", "demasiado", "no", "permitio"
}

PALABRAS_CATEGORIA = {
    "docente": {"profesor", "docente", "explica", "explicacion", "dudas", "retroalimentacion", "calificar"},
    "contenido": {"clase", "materia", "contenido", "conceptos", "tema", "temas", "teoria", "practica", "practicas", "actividad", "material"},
    "plataforma": {"plataforma", "sistema", "archivo", "entrega", "subir", "cayo"},
}

print("Diccionarios cargados correctamente.")

Diccionarios cargados correctamente.


## 3. Funciones del pipeline

In [3]:
def cargar_comentarios(ruta: Path) -> list[str]:
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo de entrada: {ruta}")
    comentarios = [linea.strip() for linea in ruta.read_text(encoding="utf-8").splitlines()]
    return [comentario for comentario in comentarios if comentario]


def quitar_acentos(texto: str) -> str:
    normalizado = unicodedata.normalize("NFD", texto)
    return "".join(c for c in normalizado if unicodedata.category(c) != "Mn")


def limpiar_texto(texto: str) -> str:
    texto = texto.lower()
    texto = quitar_acentos(texto)
    texto = re.sub(r"[^a-zñ0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def tokenizar(texto_limpio: str) -> list[str]:
    return texto_limpio.split()


def quitar_stopwords(tokens: list[str]) -> list[str]:
    return [token for token in tokens if token not in STOPWORDS and len(token) > 2]


def clasificar_sentimiento(tokens: list[str]) -> str:
    positivos = sum(1 for token in tokens if token in PALABRAS_POSITIVAS)
    negativos = sum(1 for token in tokens if token in PALABRAS_NEGATIVAS)
    if positivos > 0 and negativos > 0:
        return "mixto"
    if positivos > negativos:
        return "positivo"
    if negativos > positivos:
        return "negativo"
    return "neutral"


def clasificar_categoria(tokens: list[str]) -> str:
    puntajes = {}
    for categoria, palabras in PALABRAS_CATEGORIA.items():
        puntajes[categoria] = sum(1 for token in tokens if token in palabras)
    mejor_categoria, mejor_puntaje = max(puntajes.items(), key=lambda item: item[1])
    return mejor_categoria if mejor_puntaje > 0 else "sin_categoria"


def analizar_comentarios(comentarios: list[str]) -> list[dict[str, str]]:
    resultados = []
    for i, comentario in enumerate(comentarios, start=1):
        limpio = limpiar_texto(comentario)
        tokens = tokenizar(limpio)
        tokens_utiles = quitar_stopwords(tokens)
        frecuencia = Counter(tokens_utiles)
        resultados.append({
            "id": str(i),
            "comentario_original": comentario,
            "texto_limpio": limpio,
            "tokens_utiles": ", ".join(tokens_utiles),
            "palabras_frecuentes": ", ".join([f"{p}:{c}" for p, c in frecuencia.most_common(5)]),
            "categoria": clasificar_categoria(tokens_utiles),
            "sentimiento": clasificar_sentimiento(tokens_utiles),
        })
    return resultados

print("Funciones definidas correctamente.")

Funciones definidas correctamente.


## 4. Carga y procesamiento de comentarios

In [4]:
comentarios = cargar_comentarios(ARCHIVO_ENTRADA)
print(f"Se cargaron {len(comentarios)} comentarios.\n")

for i, c in enumerate(comentarios, 1):
    print(f"  {i:2d}. {c}")

Se cargaron 20 comentarios.

   1. El profesor explica muy bien, pero deja demasiadas tareas.
   2. La clase es interesante, aunque a veces va muy rápido.
   3. No entendí bien los temas porque faltaron ejemplos prácticos.
   4. Excelente materia, me gustó trabajar con casos reales.
   5. El docente domina el tema, pero debería organizar mejor los tiempos.
   6. La plataforma falló varias veces durante la entrega.
   7. Me gustaría que hubiera más prácticas y menos teoría.
   8. El contenido es útil, pero algunos conceptos fueron confusos.
   9. El profesor resolvió dudas con paciencia y claridad.
  10. La actividad fue buena, aunque el tiempo no alcanzó.
  11. No me gustó que la explicación fuera demasiado rápida.
  12. La plataforma no permitió subir el archivo a tiempo.
  13. El tema fue relevante para mi trabajo profesional.
  14. La clase tuvo mucha teoría y poca aplicación.
  15. La práctica ayudó a entender mejor el concepto.
  16. El docente fue claro, pero faltó retroalimentac

In [5]:
resultados = analizar_comentarios(comentarios)

for fila in resultados:
    print("=" * 90)
    print(f"ID: {fila['id']}")
    print(f"Comentario : {fila['comentario_original']}")
    print(f"Limpio     : {fila['texto_limpio']}")
    print(f"Tokens     : {fila['tokens_utiles']}")
    print(f"Categoría  : {fila['categoria']}")
    print(f"Sentimiento: {fila['sentimiento']}")
print("=" * 90)

ID: 1
Comentario : El profesor explica muy bien, pero deja demasiadas tareas.
Limpio     : el profesor explica muy bien pero deja demasiadas tareas
Tokens     : profesor, explica, bien, deja, demasiadas, tareas
Categoría  : docente
Sentimiento: mixto
ID: 2
Comentario : La clase es interesante, aunque a veces va muy rápido.
Limpio     : la clase es interesante aunque a veces va muy rapido
Tokens     : clase, interesante, veces, rapido
Categoría  : contenido
Sentimiento: mixto
ID: 3
Comentario : No entendí bien los temas porque faltaron ejemplos prácticos.
Limpio     : no entendi bien los temas porque faltaron ejemplos practicos
Tokens     : entendi, bien, temas, porque, faltaron, ejemplos, practicos
Categoría  : contenido
Sentimiento: mixto
ID: 4
Comentario : Excelente materia, me gustó trabajar con casos reales.
Limpio     : excelente materia me gusto trabajar con casos reales
Tokens     : excelente, materia, gusto, trabajar, casos, reales
Categoría  : contenido
Sentimiento: positivo
I

## 5. Resultados en tabla y guardado en CSV

In [6]:
df = pd.DataFrame(resultados)
display(df[["id", "comentario_original", "categoria", "sentimiento"]])

,id,comentario_original,categoria,sentimiento
0,1,"El profesor explica muy bien, pero deja demasi...",docente,mixto
1,2,"La clase es interesante, aunque a veces va muy...",contenido,mixto
2,3,No entendí bien los temas porque faltaron ejem...,contenido,mixto
3,4,"Excelente materia, me gustó trabajar con casos...",contenido,positivo
4,5,"El docente domina el tema, pero debería organi...",docente,neutral
5,6,La plataforma falló varias veces durante la en...,plataforma,negativo
6,7,Me gustaría que hubiera más prácticas y menos ...,contenido,neutral
7,8,"El contenido es útil, pero algunos conceptos f...",contenido,mixto
8,9,El profesor resolvió dudas con paciencia y cla...,docente,positivo
9,10,"La actividad fue buena, aunque el tiempo no al...",contenido,positivo


In [7]:
# Guardar resultados en CSV
campos = ["id", "comentario_original", "texto_limpio", "tokens_utiles",
          "palabras_frecuentes", "categoria", "sentimiento"]

with ARCHIVO_SALIDA.open("w", encoding="utf-8-sig", newline="") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=campos)
    escritor.writeheader()
    escritor.writerows(resultados)

print(f"Resultados guardados en: {ARCHIVO_SALIDA.resolve()}")

Resultados guardados en: /Users/victor/Library/Mobile Documents/com~apple~CloudDocs/Maestría UMPH/Cuatrimestre 3/Procesamiento De Lenguaje Natural/Tarea 1/resultados_clase1.csv


## 6. Error común documentado y corregido\n\n### Error encontrado: la palabra `"no"` estaba en las stopwords Y en las palabras negativas al mismo tiempo.\n\nEn el script original, la palabra `"no"` aparece en el conjunto `STOPWORDS`, lo cual provoca que sea **eliminada** durante la fase de filtrado con `quitar_stopwords()`. Sin embargo, `"no"` también aparece en `PALABRAS_NEGATIVAS`, donde debería contribuir a detectar sentimiento negativo.\n\n**Consecuencia:** comentarios como *"No me gustó que la explicación fuera demasiado rápida"* pierden la negación `"no"` antes de llegar al clasificador de sentimiento, lo que reduce la capacidad del sistema para detectar opiniones negativas.\n\nAdemás, la función `quitar_stopwords()` filtra tokens con `len(token) > 2`, lo que también elimina `"no"` (2 caracteres). Esto genera un **doble filtrado** que asegura que la negación nunca llegue al clasificador.\n\n### Corrección aplicada:\nSe elimina `"no"` del conjunto de `STOPWORDS` para que la negación se conserve y pueda ser detectada por el clasificador de sentimiento. También se ajusta el filtro de longitud mínima a `>= 2` para no descartar palabras cortas pero significativas como `"no"`.

In [8]:
# --- Demostración del error ---
ejemplo = "No me gustó que la explicación fuera demasiado rápida."
limpio = limpiar_texto(ejemplo)
tokens = tokenizar(limpio)
tokens_filtrados_original = [t for t in tokens if t not in STOPWORDS and len(t) > 2]

print("Comentario original:", ejemplo)
print("Tokens después de limpieza:", tokens)
print("Tokens con filtro ORIGINAL (se pierde 'no'):", tokens_filtrados_original)
print("Sentimiento con filtro original:", clasificar_sentimiento(tokens_filtrados_original))

print("\n--- Aplicando corrección ---")
# Corregir: quitar "no" de STOPWORDS y ajustar longitud mínima
STOPWORDS_CORREGIDAS = STOPWORDS - {"no"}

def quitar_stopwords_corregido(tokens: list[str]) -> list[str]:
    return [token for token in tokens if token not in STOPWORDS_CORREGIDAS and len(token) >= 2]

tokens_filtrados_corregido = quitar_stopwords_corregido(tokens)
print("Tokens con filtro CORREGIDO (se conserva 'no'):", tokens_filtrados_corregido)
print("Sentimiento con filtro corregido:", clasificar_sentimiento(tokens_filtrados_corregido))

Comentario original: No me gustó que la explicación fuera demasiado rápida.
Tokens después de limpieza: ['no', 'me', 'gusto', 'que', 'la', 'explicacion', 'fuera', 'demasiado', 'rapida']
Tokens con filtro ORIGINAL (se pierde 'no'): ['gusto', 'explicacion', 'fuera', 'demasiado', 'rapida']
Sentimiento con filtro original: mixto

--- Aplicando corrección ---
Tokens con filtro CORREGIDO (se conserva 'no'): ['no', 'gusto', 'explicacion', 'fuera', 'demasiado', 'rapida']
Sentimiento con filtro corregido: mixto


## 7. Reflexión breve: ¿Qué necesita mejorar este sistema para ser útil?\n\n1. **Manejo de negaciones:** El sistema no entiende contexto. Una frase como *"no me gustó"* debería invertir el sentimiento de la palabra *"gustó"*, pero actualmente cada token se evalúa de forma aislada. Se necesitaría analizar bigramas (pares de palabras) o usar modelos más avanzados que consideren la secuencia.\n\n2. **Diccionarios más completos:** Las listas de palabras positivas y negativas son muy limitadas. Muchos comentarios quedan clasificados como "neutral" simplemente porque sus palabras clave no están en los diccionarios. Un léxico de sentimiento más amplio (o un modelo pre-entrenado) mejoraría la cobertura.\n\n3. **Manejo de sinónimos y variaciones:** Palabras como "gustó" (verbo conjugado) no se detectan porque el diccionario solo tiene "gusto". Un proceso de lematización (reducir palabras a su forma base) permitiría agrupar variaciones de una misma palabra.\n\n4. **Clasificación multi-categoría:** Actualmente cada comentario se asigna a una sola categoría, pero muchos comentarios hablan de múltiples temas (e.g., docente y contenido). Permitir etiquetas múltiples daría una visión más completa.\n\n5. **Escalabilidad:** Para un volumen grande de comentarios, un enfoque basado en diccionarios no escala. Sería preferible usar técnicas de machine learning (Naive Bayes, SVM) o modelos de lenguaje pre-entrenados (BERT en español) para clasificación automática más precisa.